# PRISM Rebuttal - Phase 1: Corrected Re-computation

Fixes applied relative to the submitted version:

1. **OOD temperature scaling leakage** - T is now fit on a held-out
   *source-domain* validation split, never on target test labels.
2. **ECE binning** - a single consistent scheme everywhere, reported under
   BOTH fixed-width and adaptive (equal-mass) binning.
3. **F1 metric** - macro-F1 everywhere (submitted version used sklearn's
   binary default on PCam / MHIST).
4. **Sampling** - class-stratified in every setting (submitted version used
   class-agnostic random sampling in-distribution).
5. **Degeneracy audit** - explicitly reports cells where the probe collapses
   to a single predicted class.
6. **Statistics** - Spearman / Kendall rank correlation between AUROC and ECE
   rankings with bootstrap CIs, replacing the rank-1 agreement count.

Outputs are written to `PRISM/results_v2/` so the original CSVs are preserved
for a side-by-side diff.

**Runtime:** CPU only, no GPU. Select **High-RAM** under
Runtime > Change runtime type.

**Crash-safe:** every (model, dataset) cell is checkpointed to disk. If the
session dies, simply re-run the failing cell; completed work is skipped.
Embeddings are memory-mapped and inference is chunked, so peak RAM stays flat
regardless of dataset size.

In [2]:
import os, itertools, warnings
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score, brier_score_loss
from scipy.optimize import minimize_scalar
from scipy.stats import spearmanr, kendalltau
from google.colab import drive

warnings.filterwarnings('ignore')
drive.mount('/content/drive')

BASE       = '/content/drive/MyDrive/PRISM'
EMB_DIR    = f'{BASE}/embeddings'
OUT_DIR    = f'{BASE}/results_v2'
OLD_DIR    = f'{BASE}/results'
os.makedirs(OUT_DIR, exist_ok=True)

MODELS = ['CLIP','PLIP','CONCH','VIRCHOW2','UNI','GigaPath','H-Optimus-0','MIDNIGHT']
MKEYS  = ['clip','plip','conch','virchow2','uni','gigapath','h_optimus_0','midnight']
M2K    = dict(zip(MODELS, MKEYS))

DATASETS = ['PCam','BRACS','CRC','MHIST','LungHist700','SPIDER-Breast']
DKEYS    = ['pcam','bracs','crc','mhist','lunghist700','spider_breast']
D2K      = dict(zip(DATASETS, DKEYS))

FRACTIONS = [0.01, 0.05, 0.10, 0.25, 0.50, 1.00]
SEEDS     = [42, 123, 456]

N_BINS    = 15          # single consistent bin count everywhere
C_DEFAULT = 1.0
MAX_ITER  = 1000

# OOD binarization (sorted folder order, verified against dataset dirs)
# CRC : ADI=0 BACK=1 DEB=2 LYM=3 MUC=4 MUS=5 NORM=6 STR=7 TUM=8
# BRACS: ADH=0 DCIS=1 FEA=2 IC=3 N=4 PB=5 UDH=6
CRC_BIN   = lambda y: (y == 8).astype(int)
BRACS_BIN = lambda y: np.isin(y, [1, 3]).astype(int)

OOD_PAIRS = [('PCam','MHIST'), ('MHIST','PCam'),
             ('CRC','BRACS'),  ('BRACS','CRC')]

print('Setup OK. Writing to', OUT_DIR)

# ---- memory-lean infrastructure -------------------------------------------
import gc, glob
CKPT_ID  = f'{OUT_DIR}/indomain_parts'
CKPT_OOD = f'{OUT_DIR}/ood_parts'
os.makedirs(CKPT_ID, exist_ok=True)
os.makedirs(CKPT_OOD, exist_ok=True)

def load_emb_mm(mkey, dkey, split):
    """Memory-mapped: only indexed rows are materialised."""
    p = f'{EMB_DIR}/{mkey}/{dkey}'
    return (np.load(f'{p}/{split}_features.npy', mmap_mode='r'),
            np.load(f'{p}/{split}_labels.npy').astype(int))

def proba_chunked(clf, X, chunk=20000):
    return np.vstack([clf.predict_proba(np.asarray(X[i:i+chunk], dtype=np.float32))
                      for i in range(0, X.shape[0], chunk)])

def logits_chunked(clf, X, chunk=20000):
    out = []
    for i in range(0, X.shape[0], chunk):
        d = clf.decision_function(np.asarray(X[i:i+chunk], dtype=np.float32))
        if d.ndim == 1:
            d = d.reshape(-1, 1); d = np.hstack([-d, d])
        out.append(d)
    return np.vstack(out)

# smallest datasets first, PCam last
ORDER = ['LungHist700', 'MHIST', 'BRACS', 'CRC', 'SPIDER-Breast', 'PCam']

print('Memory-lean helpers ready. Checkpoints ->', CKPT_ID)

Mounted at /content/drive
Setup OK. Writing to /content/drive/MyDrive/PRISM/results_v2
Memory-lean helpers ready. Checkpoints -> /content/drive/MyDrive/PRISM/results_v2/indomain_parts


## 1. Metrics

In [3]:
def _ece_from_conf(conf, correct, bin_edges):
    """Weighted |accuracy - confidence| over the supplied bin edges."""
    ece, n = 0.0, len(conf)
    for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
        m = (conf >= lo) & (conf < hi)
        if m.sum() > 0:
            ece += m.sum() * abs(correct[m].mean() - conf[m].mean())
    return float(ece / n)


def ece_fixed(conf, correct, n_bins=N_BINS):
    """Fixed-width binning (what the submitted code actually did)."""
    return _ece_from_conf(conf, correct, np.linspace(0, 1, n_bins + 1))


def ece_adaptive(conf, correct, n_bins=N_BINS):
    """Equal-mass (adaptive) binning, following Nixon et al. 2019.
    Avoids the empty-bin pathology on small test splits."""
    qs = np.linspace(0, 1, n_bins + 1)
    edges = np.quantile(conf, qs)
    edges[0], edges[-1] = 0.0, 1.0 + 1e-9
    edges = np.unique(edges)          # collapse duplicates on degenerate conf
    if len(edges) < 3:
        return ece_fixed(conf, correct, n_bins)
    return _ece_from_conf(conf, correct, edges)


def conf_and_correct(proba, y_true):
    """Turn a (N, K) probability matrix into (confidence, correctness).
    For K=2 we use p(class 1) vs the label, matching the submitted
    binary convention; for K>2 we use max-probability vs argmax accuracy."""
    if proba.shape[1] == 2:
        return proba[:, 1], (y_true == 1).astype(float)
    return proba.max(axis=1), (proba.argmax(axis=1) == y_true).astype(float)


def softmax(z):
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


def logits_matrix(clf, X):
    """Always return an (N, K) logit matrix, binary included."""
    d = clf.decision_function(X)
    if d.ndim == 1:
        d = d.reshape(-1, 1)
        d = np.hstack([-d, d])
    return d


def fit_temperature(val_logits, val_labels, bounds=(0.1, 10.0)):
    """Scalar T minimizing NLL on the SUPPLIED split.
    Callers must pass a held-out split - never the evaluation set."""
    def nll(T):
        p = softmax(val_logits / T)
        idx = np.arange(len(val_labels))
        return -np.log(p[idx, val_labels] + 1e-12).mean()
    return float(minimize_scalar(nll, bounds=bounds, method='bounded').x)


def stratified_sample(labels, fraction, seed):
    """Class-stratified subset. Guarantees >=1 sample per present class and
    records whether that guarantee had to be invoked (a degeneracy signal)."""
    np.random.seed(seed)
    idx_all = np.arange(len(labels))
    picked, forced = [], []
    for c in np.unique(labels):
        c_idx = idx_all[labels == c]
        n_exact = len(c_idx) * fraction
        n = max(1, int(n_exact))
        if n_exact < 1:
            forced.append(int(c))
        picked.extend(np.random.choice(c_idx, size=n, replace=False))
    return np.array(sorted(picked)), forced


def degeneracy(y_pred, n_classes):
    """Share of predictions falling in the single most-predicted class."""
    counts = np.bincount(y_pred, minlength=n_classes)
    return float(counts.max() / counts.sum()), int(counts.argmax())


print('Metrics ready.')

Metrics ready.


## 2. Embedding loader

In [4]:
def load_all_mm(mkey, dkey):
    """Memory-mapped train/val/test. val may be absent for some cells."""
    Xtr, ytr = load_emb_mm(mkey, dkey, 'train')
    Xte, yte = load_emb_mm(mkey, dkey, 'test')
    try:
        Xva, yva = load_emb_mm(mkey, dkey, 'val')
    except FileNotFoundError:
        Xva, yva = None, None
    return Xtr, ytr, Xva, yva, Xte, yte


print('Checking embedding availability and shapes...')
missing = []
for m, mk in zip(MODELS, MKEYS):
    for d, dk in zip(DATASETS, DKEYS):
        for s in ['train', 'val', 'test']:
            f = f'{EMB_DIR}/{mk}/{dk}/{s}_features.npy'
            if not os.path.exists(f):
                missing.append(f'{mk}/{dk}/{s}')
print(f'Missing splits: {len(missing)}')
for x in missing[:25]:
    print('  ', x)

print('\nEmbedding dimensionality per model (from PCam train):')
for m, mk in zip(MODELS, MKEYS):
    try:
        X, _ = load_emb_mm(mk, 'pcam', 'train')
        print(f'  {m:>12}: {X.shape[1]:>5}d   ({X.shape[0]:,} rows, {X.dtype})')
        del X
    except Exception as e:
        print(f'  {m:>12}: unavailable ({e})')
gc.collect()

Checking embedding availability and shapes...
Missing splits: 0

Embedding dimensionality per model (from PCam train):
          CLIP:   768d   (262,144 rows, float32)
          PLIP:   768d   (262,144 rows, float32)
         CONCH:   512d   (262,144 rows, float32)
      VIRCHOW2:  2560d   (262,144 rows, float32)
           UNI:  1024d   (262,144 rows, float32)
      GigaPath:  1536d   (262,144 rows, float32)
   H-Optimus-0:  1536d   (262,144 rows, float32)
      MIDNIGHT:  1536d   (262,144 rows, float32)


176

## 3. In-distribution re-run

Changes vs. submitted: stratified sampling, macro-F1, one consistent bin
count, both binning schemes, degeneracy audit. Temperature is fit on the
validation split (this part was already correct in the submitted code and is
preserved unchanged).

In [5]:
def run_indomain(model, dataset):
    mk, dk = M2K[model], D2K[dataset]
    Xtr, ytr, Xva, yva, Xte, yte = load_all_mm(mk, dk)
    n_classes = len(np.unique(ytr))
    rows = []

    for frac in FRACTIONS:
        for seed in SEEDS:
            idx, forced = stratified_sample(ytr, frac, seed)
            Xs = np.asarray(Xtr[idx], dtype=np.float32)
            clf = LogisticRegression(max_iter=MAX_ITER, C=C_DEFAULT,
                                     random_state=seed).fit(Xs, ytr[idx])
            del Xs; gc.collect()

            proba = proba_chunked(clf, Xte)
            pred  = proba.argmax(axis=1)
            conf, corr = conf_and_correct(proba, yte)

            try:
                auroc = (roc_auc_score(yte, proba[:, 1]) if n_classes == 2
                         else roc_auc_score(yte, proba, multi_class='ovr',
                                            average='macro'))
            except Exception:
                auroc = np.nan

            # temperature on the held-out validation split (never on test)
            if Xva is not None:
                T = fit_temperature(logits_chunked(clf, Xva), yva)
                sp = softmax(logits_chunked(clf, Xte) / T)
                s_conf, s_corr = conf_and_correct(sp, yte)
                ece_s_fix = ece_fixed(s_conf, s_corr)
                ece_s_ada = ece_adaptive(s_conf, s_corr)
                del sp
            else:
                T = ece_s_fix = ece_s_ada = np.nan

            deg_share, _ = degeneracy(pred, n_classes)

            rows.append(dict(
                model=model, dataset=dataset, fraction=frac, seed=seed,
                n_train=len(idx), n_classes=n_classes, auroc=auroc,
                f1_macro=f1_score(yte, pred, average='macro', zero_division=0),
                f1_binary=(f1_score(yte, pred, zero_division=0)
                           if n_classes == 2 else np.nan),
                brier=(brier_score_loss(yte, proba[:, 1])
                       if n_classes == 2 else np.nan),
                ece_fixed=ece_fixed(conf, corr),
                ece_adaptive=ece_adaptive(conf, corr),
                temperature=T,
                ece_scaled_fixed=ece_s_fix,
                ece_scaled_adaptive=ece_s_ada,
                degeneracy_share=deg_share,
                degenerate=deg_share > 0.99,
                forced_classes=len(forced)))
            del proba, clf; gc.collect()

    del Xtr, Xte, Xva; gc.collect()
    return pd.DataFrame(rows)


print('run_indomain ready (memory-lean, checkpointed).')

run_indomain ready (memory-lean, checkpointed).


In [6]:
for dataset in ORDER:
    for model in MODELS:
        out = f'{CKPT_ID}/{M2K[model]}__{D2K[dataset]}.csv'
        if os.path.exists(out):
            print(f'  skip (already done): {model} x {dataset}')
            continue
        try:
            df = run_indomain(model, dataset)
            df.to_csv(out, index=False)
            s = df.groupby('fraction')['auroc'].mean()
            flag = ' [DEGENERATE]' if df['degenerate'].any() else ''
            print(f'{model:>12} x {dataset:<14} '
                  f'1%={s.loc[0.01]:.4f}  100%={s.loc[1.00]:.4f}{flag}')
        except Exception as e:
            print(f'{model:>12} x {dataset:<14} FAILED: {type(e).__name__}: {e}')
        gc.collect()

parts = sorted(glob.glob(f'{CKPT_ID}/*.csv'))
df_id = pd.concat([pd.read_csv(p) for p in parts], ignore_index=True)
df_id.to_csv(f'{OUT_DIR}/indomain_all_v2.csv', index=False)
print(f'\n{len(parts)}/48 cells complete, {len(df_id)} rows -> indomain_all_v2.csv')
print('\nIf the session crashed, just re-run this cell: finished cells are skipped.')

  skip (already done): CLIP x LungHist700
  skip (already done): PLIP x LungHist700
  skip (already done): CONCH x LungHist700
  skip (already done): VIRCHOW2 x LungHist700
  skip (already done): UNI x LungHist700
  skip (already done): GigaPath x LungHist700
  skip (already done): H-Optimus-0 x LungHist700
  skip (already done): MIDNIGHT x LungHist700
  skip (already done): CLIP x MHIST
  skip (already done): PLIP x MHIST
  skip (already done): CONCH x MHIST
  skip (already done): VIRCHOW2 x MHIST
  skip (already done): UNI x MHIST
  skip (already done): GigaPath x MHIST
  skip (already done): H-Optimus-0 x MHIST
  skip (already done): MIDNIGHT x MHIST
  skip (already done): CLIP x BRACS
  skip (already done): PLIP x BRACS
  skip (already done): CONCH x BRACS
  skip (already done): VIRCHOW2 x BRACS
  skip (already done): UNI x BRACS
  skip (already done): GigaPath x BRACS
  skip (already done): H-Optimus-0 x BRACS
  skip (already done): MIDNIGHT x BRACS
  skip (already done): CLIP x C

## 4. OOD re-run, leakage-free

The submitted OOD code fit the temperature on the target test labels:

```python
T = temperature_scale_binary(logits, tgt_int)   # target test labels
```

Here T is fit on a held-out source-domain validation split and applied to the
target without ever touching target labels. We report the original
target-fitted value as `temperature_oracle` / `ece_scaled_oracle` alongside,
so the two can be compared directly in the rebuttal.

In [7]:
def load_ood_side(mk, name):
    dk = D2K[name]
    Xtr, ytr, Xva, yva, Xte, yte = load_all(mk, dk)
    if name == 'CRC':
        ytr, yva, yte = CRC_BIN(ytr), (None if yva is None else CRC_BIN(yva)), CRC_BIN(yte)
    elif name == 'BRACS':
        ytr, yva, yte = BRACS_BIN(ytr), (None if yva is None else BRACS_BIN(yva)), BRACS_BIN(yte)
    return Xtr, ytr, Xva, yva, Xte, yte


def run_ood(model, src, tgt):
    mk = M2K[model]
    Xs_tr, ys_tr, Xs_va, ys_va, _, _ = load_ood_side(mk, src)
    _, _, _, _, Xt_te, yt_te = load_ood_side(mk, tgt)
    rows = []

    for frac in FRACTIONS:
        for seed in SEEDS:
            idx, _ = stratified_sample(ys_tr, frac, seed)
            clf = LogisticRegression(max_iter=MAX_ITER, C=C_DEFAULT,
                                     random_state=seed)
            clf.fit(Xs_tr[idx], ys_tr[idx])

            proba = clf.predict_proba(Xt_te)
            pred  = proba.argmax(axis=1)
            conf, corr = conf_and_correct(proba, yt_te)

            try:
                auroc = roc_auc_score(yt_te, proba[:, 1])
            except Exception:
                auroc = np.nan

            Lt = logits_matrix(clf, Xt_te)

            # CORRECTED: temperature from source-domain validation
            if Xs_va is not None:
                T_src = fit_temperature(logits_matrix(clf, Xs_va), ys_va)
                sp = softmax(Lt / T_src)
                c2, k2 = conf_and_correct(sp, yt_te)
                ece_s_fix, ece_s_ada = ece_fixed(c2, k2), ece_adaptive(c2, k2)
            else:
                T_src = np.nan
                ece_s_fix = ece_s_ada = np.nan

            # ORACLE (what the submitted code did) - kept for comparison
            T_or = fit_temperature(Lt, yt_te)
            sp_or = softmax(Lt / T_or)
            c3, k3 = conf_and_correct(sp_or, yt_te)

            deg_share, _ = degeneracy(pred, 2)

            rows.append(dict(
                model=model, src=src, tgt=tgt, pair=f'{src}->{tgt}',
                fraction=frac, seed=seed, n_train=len(idx),
                auroc=auroc,
                f1_macro=f1_score(yt_te, pred, average='macro', zero_division=0),
                brier=brier_score_loss(yt_te, proba[:, 1]),
                ece_fixed=ece_fixed(conf, corr),
                ece_adaptive=ece_adaptive(conf, corr),
                temperature_src=T_src,
                ece_scaled_fixed=ece_s_fix,
                ece_scaled_adaptive=ece_s_ada,
                temperature_oracle=T_or,
                ece_scaled_oracle=ece_fixed(c3, k3),
                degeneracy_share=deg_share,
                degenerate=deg_share > 0.99,
            ))
    return pd.DataFrame(rows)


print('run_ood ready.')

run_ood ready.


In [8]:
def load_ood_side(mk, name):
    dk = D2K[name]
    Xtr, ytr, Xva, yva, Xte, yte = load_all_mm(mk, dk)
    if name == 'CRC':
        ytr, yte = CRC_BIN(ytr), CRC_BIN(yte)
        yva = None if yva is None else CRC_BIN(yva)
    elif name == 'BRACS':
        ytr, yte = BRACS_BIN(ytr), BRACS_BIN(yte)
        yva = None if yva is None else BRACS_BIN(yva)
    return Xtr, ytr, Xva, yva, Xte, yte


def run_ood(model, src, tgt):
    mk = M2K[model]
    Xs_tr, ys_tr, Xs_va, ys_va, _, _ = load_ood_side(mk, src)
    _, _, _, _, Xt_te, yt_te = load_ood_side(mk, tgt)
    rows = []

    for frac in FRACTIONS:
        for seed in SEEDS:
            idx, _ = stratified_sample(ys_tr, frac, seed)
            Xs = np.asarray(Xs_tr[idx], dtype=np.float32)
            clf = LogisticRegression(max_iter=MAX_ITER, C=C_DEFAULT,
                                     random_state=seed).fit(Xs, ys_tr[idx])
            del Xs; gc.collect()

            proba = proba_chunked(clf, Xt_te)
            pred  = proba.argmax(axis=1)
            conf, corr = conf_and_correct(proba, yt_te)

            try:
                auroc = roc_auc_score(yt_te, proba[:, 1])
            except Exception:
                auroc = np.nan

            Lt = logits_chunked(clf, Xt_te)

            # CORRECTED: temperature from the SOURCE validation split
            if Xs_va is not None:
                T_src = fit_temperature(logits_chunked(clf, Xs_va), ys_va)
                sp = softmax(Lt / T_src)
                c2, k2 = conf_and_correct(sp, yt_te)
                ece_s_fix, ece_s_ada = ece_fixed(c2, k2), ece_adaptive(c2, k2)
                del sp
            else:
                T_src = ece_s_fix = ece_s_ada = np.nan

            # ORACLE: what the submitted code did (T fit on target test)
            T_or  = fit_temperature(Lt, yt_te)
            sp_or = softmax(Lt / T_or)
            c3, k3 = conf_and_correct(sp_or, yt_te)
            ece_or = ece_fixed(c3, k3)
            del sp_or, Lt; gc.collect()

            deg_share, _ = degeneracy(pred, 2)

            rows.append(dict(
                model=model, src=src, tgt=tgt, pair=f'{src}->{tgt}',
                fraction=frac, seed=seed, n_train=len(idx), auroc=auroc,
                f1_macro=f1_score(yt_te, pred, average='macro', zero_division=0),
                brier=brier_score_loss(yt_te, proba[:, 1]),
                ece_fixed=ece_fixed(conf, corr),
                ece_adaptive=ece_adaptive(conf, corr),
                temperature_src=T_src,
                ece_scaled_fixed=ece_s_fix,
                ece_scaled_adaptive=ece_s_ada,
                temperature_oracle=T_or,
                ece_scaled_oracle=ece_or,
                degeneracy_share=deg_share,
                degenerate=deg_share > 0.99))
            del proba, clf; gc.collect()

    del Xs_tr, Xs_va, Xt_te; gc.collect()
    return pd.DataFrame(rows)


print('run_ood ready (memory-lean, leakage-free + oracle comparison).')

run_ood ready (memory-lean, leakage-free + oracle comparison).


In [11]:
for model in MODELS:
    for src, tgt in OOD_PAIRS:
        out = f'{CKPT_OOD}/{M2K[model]}__{D2K[src]}to{D2K[tgt]}.csv'
        if os.path.exists(out):
            print(f' skip (already done): {model} {src}->{tgt}')
            continue
        try:
            df = run_ood(model, src, tgt)
            df.to_csv(out, index=False)
            s = df.groupby('fraction')[['ece_fixed','ece_scaled_fixed', 'ece_scaled_oracle']].mean()
            print(f'{model:>12} {src:>6}->{tgt:<6} ' f'ECE_raw 1%={s.loc[0.01,"ece_fixed"]:.3f} ' f'100%={s.loc[1.00,"ece_fixed"]:.3f} | ' f'scaled(src)={s.loc[1.00,"ece_scaled_fixed"]:.3f} ' f'(oracle {s.loc[1.00,"ece_scaled_oracle"]:.3f})')
        except Exception as e:
            print(f'{model:>12} {src:>6}->{tgt:<6} FAILED: {type(e).name}: {e}')
        gc.collect()

parts = sorted(glob.glob(f'{CKPT_OOD}/*.csv'))
df_ood = pd.concat([pd.read_csv(p) for p in parts], ignore_index=True)
df_ood.to_csv(f'{OUT_DIR}/ood_all_v2.csv', index=False)
print(f'\n{len(parts)}/32 pairs complete, {len(df_ood)} rows -> ood_all_v2.csv')

        CLIP   PCam->MHIST  ECE_raw 1%=0.208 100%=0.251 | scaled(src)=0.246 (oracle 0.096)
        CLIP  MHIST->PCam   ECE_raw 1%=0.234 100%=0.312 | scaled(src)=0.369 (oracle 0.100)
        CLIP    CRC->BRACS  ECE_raw 1%=0.078 100%=0.204 | scaled(src)=0.251 (oracle 0.198)
        CLIP  BRACS->CRC    ECE_raw 1%=0.215 100%=0.300 | scaled(src)=0.288 (oracle 0.320)
        PLIP   PCam->MHIST  ECE_raw 1%=0.324 100%=0.261 | scaled(src)=0.250 (oracle 0.137)
        PLIP  MHIST->PCam   ECE_raw 1%=0.216 100%=0.207 | scaled(src)=0.263 (oracle 0.046)
        PLIP    CRC->BRACS  ECE_raw 1%=0.073 100%=0.179 | scaled(src)=0.219 (oracle 0.063)
        PLIP  BRACS->CRC    ECE_raw 1%=0.183 100%=0.261 | scaled(src)=0.269 (oracle 0.258)
       CONCH   PCam->MHIST  ECE_raw 1%=0.112 100%=0.151 | scaled(src)=0.144 (oracle 0.043)
       CONCH  MHIST->PCam   ECE_raw 1%=0.229 100%=0.455 | scaled(src)=0.467 (oracle 0.191)
       CONCH    CRC->BRACS  ECE_raw 1%=0.239 100%=0.338 | scaled(src)=0.350 (oracle 0.194)

In [12]:
deg_id = (df_id.groupby(['dataset','model','fraction'])['degenerate']
                .mean().reset_index())
deg_id = deg_id[deg_id['degenerate'] > 0]

print('=== In-distribution degenerate cells (single-class prediction) ===')
if len(deg_id):
    for ds in deg_id['dataset'].unique():
        sub = deg_id[deg_id['dataset'] == ds]
        print(f'\n{ds}:')
        print(sub.pivot_table(index='model', columns='fraction',
                              values='degenerate').fillna(0).round(2).to_string())
else:
    print('  none')

deg_ood = (df_ood.groupby(['pair','model','fraction'])['degenerate']
                 .mean().reset_index())
deg_ood = deg_ood[deg_ood['degenerate'] > 0]
print('\n\n=== OOD degenerate cells ===')
if len(deg_ood):
    for p in deg_ood['pair'].unique():
        sub = deg_ood[deg_ood['pair'] == p]
        print(f'\n{p}:')
        print(sub.pivot_table(index='model', columns='fraction',
                              values='degenerate').fillna(0).round(2).to_string())
else:
    print('  none')

pd.concat([deg_id.assign(scope='in-distribution'),
           deg_ood.rename(columns={'pair':'dataset'}).assign(scope='ood')]
          ).to_csv(f'{OUT_DIR}/degeneracy_audit.csv', index=False)
print('\nSaved -> degeneracy_audit.csv')

=== In-distribution degenerate cells (single-class prediction) ===

BRACS:
fraction  0.01
model         
CLIP      0.33

LungHist700:
fraction  0.05  0.10
model               
CLIP      1.00   1.0
PLIP      0.33   0.0

MHIST:
fraction     0.01  0.05  0.10  0.25
model                              
CLIP          1.0  1.00  1.00  0.67
CONCH         1.0  1.00  1.00  0.00
GigaPath      1.0  1.00  0.33  0.00
H-Optimus-0   1.0  0.67  0.00  0.00
MIDNIGHT      1.0  1.00  1.00  0.00
PLIP          1.0  1.00  1.00  0.00
UNI           1.0  1.00  0.67  0.00
VIRCHOW2      1.0  1.00  0.67  0.00


=== OOD degenerate cells ===

BRACS->CRC:
fraction     0.01  0.05  0.10  0.25
model                              
CLIP          1.0  0.00  0.00  0.00
CONCH         1.0  0.00  0.00  0.00
GigaPath      1.0  0.33  0.00  0.00
H-Optimus-0   1.0  0.67  0.67  0.00
MIDNIGHT      1.0  1.00  0.33  0.33
PLIP          1.0  0.33  0.00  0.00
UNI           1.0  1.00  0.33  0.00
VIRCHOW2      1.0  1.00  0.33  0.33

CRC->BRAC

## 6. Rank-correlation analysis

Replaces the rank-1 agreement count ("7 of 48") with a statistically grounded
measure, as requested by Reviewer tp5b. Bootstrap CIs are over models within
each (dataset, fraction) cell.

In [13]:
def rank_corr(dataset, fraction, ece_col='ece_scaled_fixed', n_boot=2000):
    sub = (df_id[(df_id.dataset == dataset) & (df_id.fraction == fraction)]
           .groupby('model')[['auroc', ece_col]].mean().dropna())
    if len(sub) < 3:
        return None
    a, e = sub['auroc'].values, sub[ece_col].values
    rho = spearmanr(a, -e).correlation      # negate: lower ECE = better
    tau = kendalltau(a, -e).correlation
    rng = np.random.default_rng(0)
    boots = []
    for _ in range(n_boot):
        i = rng.integers(0, len(a), len(a))
        if len(np.unique(a[i])) < 3:
            continue
        r = spearmanr(a[i], -e[i]).correlation
        if not np.isnan(r):
            boots.append(r)
    lo, hi = (np.percentile(boots, [2.5, 97.5]) if boots else (np.nan, np.nan))
    return dict(dataset=dataset, fraction=fraction, n_models=len(sub),
                spearman=rho, kendall=tau, ci_lo=lo, ci_hi=hi)


rc = [r for ds in DATASETS for f in FRACTIONS
      if (r := rank_corr(ds, f)) is not None]
df_rc = pd.DataFrame(rc)
df_rc.to_csv(f'{OUT_DIR}/rank_correlations.csv', index=False)

print('Spearman rho between AUROC rank and (negated) scaled-ECE rank')
print('rho ~ 0 supports the decoupling claim; rho ~ 1 refutes it.\n')
print(df_rc.pivot_table(index='dataset', columns='fraction',
                        values='spearman').round(3).to_string())
print('\nMean rho at 1% labels :', df_rc[df_rc.fraction == 0.01]['spearman'].mean().round(3))
print('Mean rho at 100% labels:', df_rc[df_rc.fraction == 1.00]['spearman'].mean().round(3))

Spearman rho between AUROC rank and (negated) scaled-ECE rank
rho ~ 0 supports the decoupling claim; rho ~ 1 refutes it.

fraction        0.01   0.05   0.10   0.25   0.50   1.00
dataset                                                
BRACS         -0.524  0.738  0.762  0.048 -0.762 -0.024
CRC            0.643  0.833  0.905  0.571  0.976  0.714
LungHist700   -0.714  0.381  0.000  0.738  0.857  0.738
MHIST         -0.786 -0.262 -0.381 -0.095  0.548  0.667
PCam          -0.571 -0.048  0.595  0.810  0.833  0.738
SPIDER-Breast  0.524  0.690  0.571  0.595  0.405  0.262

Mean rho at 1% labels : -0.238
Mean rho at 100% labels: 0.516


## 7. Regularisation sensitivity (Reviewer fWEj Q3)

Also addresses tp5b's concern that low-label estimates may be unstable.

In [14]:
C_GRID   = [0.01, 0.1, 1.0, 10.0, 100.0]
ABL_DS   = ['MHIST', 'LungHist700', 'BRACS']   # small datasets only
ABL_FRAC = [0.01, 0.10, 1.00]
ABL_OUT  = f'{OUT_DIR}/c_ablation.csv'

rows = []
for model in MODELS:
    for ds in ABL_DS:
        mk, dk = M2K[model], D2K[ds]
        try:
            Xtr, ytr, _, _, Xte, yte = load_all_mm(mk, dk)
        except Exception:
            continue
        n_classes = len(np.unique(ytr))
        for frac in ABL_FRAC:
            for C in C_GRID:
                res = []
                for seed in SEEDS:
                    idx, _ = stratified_sample(ytr, frac, seed)
                    Xs = np.asarray(Xtr[idx], dtype=np.float32)
                    clf = LogisticRegression(max_iter=MAX_ITER, C=C,
                                             random_state=seed).fit(Xs, ytr[idx])
                    del Xs
                    proba = proba_chunked(clf, Xte)
                    conf, corr = conf_and_correct(proba, yte)
                    try:
                        au = (roc_auc_score(yte, proba[:, 1]) if n_classes == 2
                              else roc_auc_score(yte, proba, multi_class='ovr',
                                                 average='macro'))
                    except Exception:
                        au = np.nan
                    res.append((au, ece_fixed(conf, corr)))
                    del proba, clf
                rows.append(dict(model=model, dataset=ds, fraction=frac, C=C,
                                 auroc=np.nanmean([r[0] for r in res]),
                                 ece=np.nanmean([r[1] for r in res])))
        del Xtr, Xte; gc.collect()
    print(f'  {model} done')

df_c = pd.DataFrame(rows)
df_c.to_csv(ABL_OUT, index=False)

print('\nAUROC spread across C (max - min), averaged over models:')
print(df_c.groupby(['dataset','fraction','model'])['auroc']
          .agg(lambda v: v.max() - v.min())
          .groupby(level=[0,1]).mean().round(4).to_string())
print('\nECE spread across C:')
print(df_c.groupby(['dataset','fraction','model'])['ece']
          .agg(lambda v: v.max() - v.min())
          .groupby(level=[0,1]).mean().round(4).to_string())

  CLIP done
  PLIP done
  CONCH done
  VIRCHOW2 done
  UNI done
  GigaPath done
  H-Optimus-0 done
  MIDNIGHT done

AUROC spread across C (max - min), averaged over models:
dataset      fraction
BRACS        0.01        0.0221
             0.10        0.0554
             1.00        0.0652
LungHist700  0.01        0.0153
             0.10        0.0480
             1.00        0.0929
MHIST        0.01        0.0294
             0.10        0.0792
             1.00        0.1092

ECE spread across C:
dataset      fraction
BRACS        0.01        0.1067
             0.10        0.1313
             1.00        0.1395
LungHist700  0.01        0.1402
             0.10        0.1659
             1.00        0.2398
MHIST        0.01        0.0655
             0.10        0.0897
             1.00        0.1129


## 8. CRI aggregation sensitivity (Reviewer fWEj Q2)

The submitted paper defines OOD_Stability as the ratio of OOD to
in-distribution AUROC. The shipped package computed something different
(1 - CV of in-distribution AUROC). This cell implements the *paper's*
definition and reports the composite under four aggregation choices.

In [15]:
# OOD_Stability per the paper: mean(OOD AUROC / ID AUROC), clipped to 1
id100 = (df_id[df_id.fraction == 1.00]
         .groupby(['model','dataset'])['auroc'].mean())

stab = {}
for model in MODELS:
    ratios = []
    for src, tgt in OOD_PAIRS:
        o = df_ood[(df_ood.model == model) & (df_ood.src == src) &
                   (df_ood.tgt == tgt) & (df_ood.fraction == 1.00)]['auroc'].mean()
        try:
            i = id100.loc[(model, src)]
        except KeyError:
            continue
        if i and not np.isnan(o):
            ratios.append(min(o / i, 1.0))
    stab[model] = float(np.mean(ratios)) if ratios else np.nan

print('OOD_Stability (paper definition):')
for m, v in sorted(stab.items(), key=lambda x: -x[1]):
    print(f'  {m:>12}: {v:.4f}')


def cri_table(fraction, ece_col='ece_scaled_fixed'):
    rows = []
    for model in MODELS:
        sub = (df_id[(df_id.model == model) & (df_id.fraction == fraction)]
               .groupby('dataset')[['auroc', ece_col]].mean())
        if sub.empty:
            continue
        a = sub['auroc'].mean()
        e = float(np.clip(sub[ece_col].mean(), 0, 1))
        s = stab.get(model, np.nan)
        rows.append(dict(
            model=model, auroc=a, ece_scaled=e, ood_stability=s,
            cri_multiplicative = a * (1 - e) * s,
            cri_arithmetic     = np.mean([a, 1 - e, s]),
            cri_geometric      = (a * (1 - e) * s) ** (1/3),
            cri_worst_axis     = min(a, 1 - e, s),
        ))
    return pd.DataFrame(rows).set_index('model')


for frac in [0.01, 1.00]:
    t = cri_table(frac)
    print(f'\n=== CRI variants at {frac:.0%} labels ===')
    print(t.round(4).to_string())
    cols = ['cri_multiplicative','cri_arithmetic','cri_geometric','cri_worst_axis']
    print('\nKendall tau between rankings:')
    for x, y in itertools.combinations(cols, 2):
        tau = kendalltau(t[x], t[y]).correlation
        print(f'  {x.replace("cri_",""):>15} vs {y.replace("cri_",""):<15} tau={tau:.3f}')
    t.to_csv(f'{OUT_DIR}/cri_variants_{int(frac*100)}.csv')

OOD_Stability (paper definition):
      VIRCHOW2: 0.7001
          CLIP: 0.6433
   H-Optimus-0: 0.6061
      GigaPath: 0.6055
          PLIP: 0.5974
      MIDNIGHT: 0.5501
         CONCH: 0.5418
           UNI: 0.5214

=== CRI variants at 1% labels ===
              auroc  ece_scaled  ood_stability  cri_multiplicative  cri_arithmetic  cri_geometric  cri_worst_axis
model                                                                                                            
CLIP         0.7787      0.0351         0.6433              0.4834          0.7956         0.7848          0.6433
PLIP         0.8463      0.0709         0.5974              0.4697          0.7909         0.7773          0.5974
CONCH        0.8755      0.0628         0.5418              0.4445          0.7848         0.7632          0.5418
VIRCHOW2     0.8662      0.0695         0.7001              0.5643          0.8323         0.8264          0.7001
UNI          0.8543      0.0557         0.5214              0.4

## 9. Old vs new comparison

What actually changed. This table is the backbone of the rebuttal: it shows
which submitted numbers survive the corrections and which move.

In [16]:
def load_old_indomain():
    NAME_FIX = {('h_optimus_0','pcam'): 'hoptimus'}
    out = []
    for model, mk in zip(MODELS, MKEYS):
        for ds, dk in zip(DATASETS, DKEYS):
            key = NAME_FIX.get((mk, dk), mk)
            p = f'{OLD_DIR}/{key}_{dk}_results.csv'
            if os.path.exists(p):
                d = pd.read_csv(p)
                d['model'], d['dataset'] = model, ds
                out.append(d)
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame()


old = load_old_indomain()
if len(old):
    o = old.groupby(['model','dataset','fraction'])[['auroc','ece']].mean()
    o.columns = ['auroc_old','ece_old']
    n = df_id.groupby(['model','dataset','fraction'])[['auroc','ece_fixed']].mean()
    n.columns = ['auroc_new','ece_new']
    cmp = o.join(n, how='inner')
    cmp['d_auroc'] = cmp['auroc_new'] - cmp['auroc_old']
    cmp['d_ece']   = cmp['ece_new']   - cmp['ece_old']
    cmp.to_csv(f'{OUT_DIR}/old_vs_new_indomain.csv')

    print('Absolute change after stratified sampling + unified binning\n')
    print('AUROC, mean |delta| by fraction:')
    print(cmp.groupby(level=2)['d_auroc'].apply(lambda v: v.abs().mean()).round(4).to_string())
    print('\nECE, mean |delta| by fraction:')
    print(cmp.groupby(level=2)['d_ece'].apply(lambda v: v.abs().mean()).round(4).to_string())
    print('\nLargest 15 AUROC shifts:')
    print(cmp.reindex(cmp['d_auroc'].abs().sort_values(ascending=False).index)
             .head(15)[['auroc_old','auroc_new','d_auroc']].round(4).to_string())
else:
    print('Original results not found at', OLD_DIR)


# headline OOD claim, before and after
print('\n\n=== Reverse OOD scaling on MHIST->PCam (raw ECE) ===')
h = (df_ood[(df_ood.pair == 'MHIST->PCam')]
     .groupby(['model','fraction'])['ece_fixed'].mean().unstack())
print(h.round(3).to_string())
print('\nMonotone increase in fraction?')
for m in h.index:
    v = h.loc[m].values
    print(f'  {m:>12}: {"YES" if all(np.diff(v) >= -1e-9) else "no "} '
          f'({v[0]:.3f} -> {v[-1]:.3f})')

Absolute change after stratified sampling + unified binning

AUROC, mean |delta| by fraction:
fraction
0.01    0.0114
0.05    0.0066
0.10    0.0049
0.25    0.0030
0.50    0.0016
1.00    0.0000

ECE, mean |delta| by fraction:
fraction
0.01    0.0189
0.05    0.0147
0.10    0.0140
0.25    0.0086
0.50    0.0037
1.00    0.0000

Largest 15 AUROC shifts:
                                  auroc_old  auroc_new  d_auroc
model       dataset     fraction                               
CONCH       MHIST       0.01         0.6864     0.6383  -0.0482
MIDNIGHT    LungHist700 0.01         0.7447     0.6970  -0.0477
VIRCHOW2    MHIST       0.01         0.6758     0.6291  -0.0467
MIDNIGHT    MHIST       0.01         0.5456     0.4992  -0.0464
GigaPath    MHIST       0.05         0.8379     0.7945  -0.0433
VIRCHOW2    LungHist700 0.01         0.7678     0.8104   0.0426
H-Optimus-0 MHIST       0.01         0.6592     0.7012   0.0420
CLIP        LungHist700 0.01         0.6351     0.5936  -0.0415
MIDNIGHT  

## 10. Done

`results_v2/` now contains:

| file | contents |
|---|---|
| `indomain_all_v2.csv` | 864 corrected in-distribution runs |
| `ood_all_v2.csv` | corrected OOD runs, leakage-free plus oracle comparison |
| `degeneracy_audit.csv` | single-class-collapse cells |
| `rank_correlations.csv` | Spearman / Kendall with bootstrap CIs |
| `c_ablation.csv` | regularisation sensitivity |
| `cri_variants_1.csv`, `cri_variants_100.csv` | four CRI aggregations |
| `old_vs_new_indomain.csv` | submitted vs corrected, per cell |

Paste the printed output back into the chat and we will write the rebuttal
against the real numbers.

In [17]:
print(pd.read_csv(f'{OUT_DIR}/rank_correlations.csv')
        .query('fraction in [0.01, 1.00]')
        [['dataset','fraction','spearman','ci_lo','ci_hi']]
        .round(3).to_string(index=False))

      dataset  fraction  spearman  ci_lo  ci_hi
         PCam      0.01    -0.571 -1.000  0.259
         PCam      1.00     0.738  0.089  1.000
        BRACS      0.01    -0.524 -1.000  0.284
        BRACS      1.00    -0.024 -0.918  0.923
          CRC      0.01     0.643 -0.063  1.000
          CRC      1.00     0.714 -0.063  1.000
        MHIST      0.01    -0.786 -1.000 -0.134
        MHIST      1.00     0.667  0.062  0.974
  LungHist700      0.01    -0.714 -1.000  0.111
  LungHist700      1.00     0.738  0.035  1.000
SPIDER-Breast      0.01     0.524 -0.407  0.975
SPIDER-Breast      1.00     0.262 -0.519  0.923


In [18]:
from scipy.stats import spearmanr, wilcoxon

ECE_COL = 'ece_scaled_fixed'

def ranks_at(fraction):
    """Within each dataset rank models by AUROC and by calibration,
    then pool. Rank 1 = best on that axis."""
    out = []
    for ds in DATASETS:
        sub = (df_id[(df_id.dataset == ds) & (df_id.fraction == fraction)]
               .groupby('model')[['auroc', ECE_COL]].mean().dropna())
        if len(sub) < 3:
            continue
        r_auc = sub['auroc'].rank(ascending=False)
        r_cal = sub[ECE_COL].rank(ascending=True)
        for m in sub.index:
            out.append(dict(dataset=ds, model=m,
                            rank_auroc=r_auc[m], rank_cal=r_cal[m],
                            rank_diff=abs(r_auc[m] - r_cal[m])))
    return pd.DataFrame(out)


def cluster_bootstrap_rho(fraction, n_boot=5000, seed=0):
    """Resample whole datasets (clusters) to respect within-dataset
    dependence between models."""
    d = ranks_at(fraction)
    dsets = d['dataset'].unique()
    rho = spearmanr(d['rank_auroc'], d['rank_cal']).correlation
    rng = np.random.default_rng(seed)
    boots = []
    for _ in range(n_boot):
        pick = rng.choice(dsets, size=len(dsets), replace=True)
        s = pd.concat([d[d.dataset == p] for p in pick], ignore_index=True)
        r = spearmanr(s['rank_auroc'], s['rank_cal']).correlation
        if not np.isnan(r):
            boots.append(r)
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return rho, lo, hi, d


print('Pooled within-dataset rank correlation (n = 48 model-dataset cells)')
print('cluster bootstrap over datasets, 95% CI\n')
print(f"{'frac':>6} {'rho':>8} {'CI low':>8} {'CI high':>8} "
      f"{'mean |rank diff|':>18} {'n':>4}")
summary = {}
for f in FRACTIONS:
    rho, lo, hi, d = cluster_bootstrap_rho(f)
    summary[f] = (rho, lo, hi, d['rank_diff'].mean())
    star = ' *' if (lo > 0 or hi < 0) else ''
    print(f'{f:>6.2f} {rho:>8.3f} {lo:>8.3f} {hi:>8.3f} '
          f'{d["rank_diff"].mean():>18.2f} {len(d):>4}{star}')

# difference between the two endpoints, cluster bootstrap on the contrast
d1, d2 = ranks_at(0.01), ranks_at(1.00)
dsets = sorted(set(d1.dataset) & set(d2.dataset))
rng = np.random.default_rng(1)
diffs = []
for _ in range(5000):
    pick = rng.choice(dsets, size=len(dsets), replace=True)
    a = pd.concat([d1[d1.dataset == p] for p in pick], ignore_index=True)
    b = pd.concat([d2[d2.dataset == p] for p in pick], ignore_index=True)
    ra = spearmanr(a['rank_auroc'], a['rank_cal']).correlation
    rb = spearmanr(b['rank_auroc'], b['rank_cal']).correlation
    if not (np.isnan(ra) or np.isnan(rb)):
        diffs.append(rb - ra)
lo, hi = np.percentile(diffs, [2.5, 97.5])
obs = summary[1.00][0] - summary[0.01][0]
print(f'\nrho(100%) - rho(1%) = {obs:.3f}   95% CI [{lo:.3f}, {hi:.3f}]'
      f'{"  SIGNIFICANT" if lo > 0 else ""}')

# paired test across datasets on the per-dataset rho
per = pd.read_csv(f'{OUT_DIR}/rank_correlations.csv')
p1 = per[per.fraction == 0.01].set_index('dataset')['spearman']
p2 = per[per.fraction == 1.00].set_index('dataset')['spearman']
common = p1.index.intersection(p2.index)
stat, pval = wilcoxon(p2[common], p1[common])
print(f'\nWilcoxon signed-rank across {len(common)} datasets: '
      f'W={stat:.1f}, p={pval:.4f}')
print(f'  rho increases from 1% to 100% in '
      f'{(p2[common] > p1[common]).sum()}/{len(common)} datasets')

# per-fraction rank-difference table, the most interpretable statistic
print('\nMean |AUROC rank - calibration rank| per dataset:')
tab = {}
for f in FRACTIONS:
    tab[f] = ranks_at(f).groupby('dataset')['rank_diff'].mean()
print(pd.DataFrame(tab).round(2).to_string())

pd.DataFrame(tab).to_csv(f'{OUT_DIR}/rank_diff_by_fraction.csv')

Pooled within-dataset rank correlation (n = 48 model-dataset cells)
cluster bootstrap over datasets, 95% CI

  frac      rho   CI low  CI high   mean |rank diff|    n
  0.01   -0.238   -0.671    0.214               2.96   48
  0.05    0.389    0.048    0.702               1.79   48 *
  0.10    0.409    0.024    0.734               1.92   48 *
  0.25    0.444    0.159    0.702               1.96   48 *
  0.50    0.476   -0.056    0.845               1.62   48
  1.00    0.516    0.266    0.722               1.79   48 *

rho(100%) - rho(1%) = 0.754   95% CI [0.190, 1.270]  SIGNIFICANT

Wilcoxon signed-rank across 6 datasets: W=2.0, p=0.0938
  rho increases from 1% to 100% in 5/6 datasets

Mean |AUROC rank - calibration rank| per dataset:
               0.01  0.05  0.10  0.25  0.50  1.00
dataset                                          
BRACS          3.25  1.00  1.25  2.50  3.50  2.50
CRC            1.75  1.00  0.75  2.00  0.25  1.50
LungHist700    4.00  2.00  2.75  1.50  1.00  1.50
MHIST

In [19]:
# Random-effects meta-analysis of per-dataset rho (Fisher z, DerSimonian-Laird)
per = pd.read_csv(f'{OUT_DIR}/rank_correlations.csv')

def meta(fraction):
    d = per[per.fraction == fraction].dropna(subset=['spearman'])
    n = d['n_models'].values                    # 8 models per dataset
    z = np.arctanh(np.clip(d['spearman'].values, -0.999, 0.999))
    v = 1.0 / (n - 3)                           # variance of Fisher z
    w = 1 / v
    mu_fe = (w * z).sum() / w.sum()
    Q  = (w * (z - mu_fe) ** 2).sum()
    df = len(z) - 1
    C  = w.sum() - (w ** 2).sum() / w.sum()
    tau2 = max(0.0, (Q - df) / C)               # between-dataset variance
    w2 = 1 / (v + tau2)
    mu = (w2 * z).sum() / w2.sum()
    se = np.sqrt(1 / w2.sum())
    lo, hi = mu - 1.96 * se, mu + 1.96 * se
    I2 = max(0.0, (Q - df) / Q) * 100 if Q > 0 else 0.0
    return np.tanh(mu), np.tanh(lo), np.tanh(hi), tau2, I2

print('Random-effects meta-analysis across datasets (Fisher z)\n')
print(f"{'frac':>6} {'rho':>8} {'CI low':>8} {'CI high':>8} {'tau2':>7} {'I2 %':>7}")
for f in FRACTIONS:
    r, lo, hi, t2, i2 = meta(f)
    star = ' *' if (lo > 0 or hi < 0) else ''
    print(f'{f:>6.2f} {r:>8.3f} {lo:>8.3f} {hi:>8.3f} {t2:>7.3f} {i2:>7.1f}{star}')

Random-effects meta-analysis across datasets (Fisher z)

  frac      rho   CI low  CI high    tau2    I2 %
  0.01   -0.298   -0.731    0.307   0.408    67.1
  0.05    0.472    0.046    0.753   0.142    41.5 *
  0.10    0.517    0.025    0.807   0.267    57.2 *
  0.25    0.508    0.169    0.740   0.037    15.6 *
  0.50    0.658   -0.072    0.929   0.958    82.7
  1.00    0.565    0.275    0.761   0.000     0.0 *


In [20]:
C_GRID   = [0.01, 0.1, 1.0, 10.0, 100.0]
ABL_DS   = ['MHIST', 'LungHist700', 'BRACS']
ABL_FRAC = [0.01, 0.10, 1.00]

rows = []
for model in MODELS:
    for ds in ABL_DS:
        mk, dk = M2K[model], D2K[ds]
        try:
            Xtr, ytr, _, _, Xte, yte = load_all_mm(mk, dk)
        except Exception as e:
            print(f'  skip {model} x {ds}: {e}'); continue
        n_classes = len(np.unique(ytr))
        for frac in ABL_FRAC:
            for C in C_GRID:
                res = []
                for seed in SEEDS:
                    idx, _ = stratified_sample(ytr, frac, seed)
                    clf = LogisticRegression(max_iter=MAX_ITER, C=C,
                              random_state=seed).fit(
                              np.asarray(Xtr[idx], dtype=np.float32), ytr[idx])
                    proba = proba_chunked(clf, Xte)
                    conf, corr = conf_and_correct(proba, yte)
                    try:
                        au = (roc_auc_score(yte, proba[:, 1]) if n_classes == 2
                              else roc_auc_score(yte, proba, multi_class='ovr',
                                                 average='macro'))
                    except Exception:
                        au = np.nan
                    res.append((au, ece_fixed(conf, corr)))
                    del proba, clf
                rows.append(dict(model=model, dataset=ds, fraction=frac, C=C,
                                 auroc=np.nanmean([r[0] for r in res]),
                                 ece=np.nanmean([r[1] for r in res])))
        del Xtr, Xte; gc.collect()
    print(f'  {model} done')

df_c = pd.DataFrame(rows)
df_c.to_csv(f'{OUT_DIR}/c_ablation.csv', index=False)

spread = lambda v: v.max() - v.min()
print('\nAUROC spread across C (max-min), averaged over models:')
print(df_c.groupby(['dataset','fraction','model'])['auroc'].agg(spread)
          .groupby(level=[0,1]).mean().round(4).to_string())
print('\nECE spread across C:')
print(df_c.groupby(['dataset','fraction','model'])['ece'].agg(spread)
          .groupby(level=[0,1]).mean().round(4).to_string())
print('\nDoes the model ranking change with C? (Kendall tau vs C=1.0)')
for ds in ABL_DS:
    for frac in ABL_FRAC:
        base = df_c.query('dataset==@ds and fraction==@frac and C==1.0') \
                   .set_index('model')['auroc']
        line = []
        for C in C_GRID:
            if C == 1.0: continue
            alt = df_c.query('dataset==@ds and fraction==@frac and C==@C') \
                      .set_index('model')['auroc']
            common = base.index.intersection(alt.index)
            line.append(f'C={C}: {kendalltau(base[common], alt[common]).correlation:.2f}')
        print(f'  {ds:<12} {frac:>5.0%}  ' + '  '.join(line))

  CLIP done
  PLIP done


KeyboardInterrupt: 

In [21]:
from scipy.stats import kendalltau
df_c = pd.read_csv(f'{OUT_DIR}/c_ablation.csv')

print('Model-ranking stability under C (Kendall tau vs C=1.0)\n')
for ds in ['MHIST','LungHist700','BRACS']:
    for frac in [0.01, 0.10, 1.00]:
        b = df_c.query('dataset==@ds and fraction==@frac and C==1.0').set_index('model')
        cells = []
        for C in [0.01, 0.1, 10.0, 100.0]:
            a = df_c.query('dataset==@ds and fraction==@frac and C==@C').set_index('model')
            k = b.index.intersection(a.index)
            ta = kendalltau(b.loc[k,'auroc'], a.loc[k,'auroc']).correlation
            te = kendalltau(b.loc[k,'ece'],   a.loc[k,'ece']).correlation
            cells.append(f'C={C:<5} auroc={ta:+.2f} ece={te:+.2f}')
        print(f'{ds:<12} {frac:>5.0%}   ' + '   '.join(cells))

Model-ranking stability under C (Kendall tau vs C=1.0)

MHIST           1%   C=0.01  auroc=+1.00 ece=+0.07   C=0.1   auroc=+1.00 ece=+0.29   C=10.0  auroc=+0.71 ece=+0.00   C=100.0 auroc=+0.43 ece=-0.64
MHIST          10%   C=0.01  auroc=+0.71 ece=-0.07   C=0.1   auroc=+0.79 ece=+0.07   C=10.0  auroc=+0.71 ece=+0.21   C=100.0 auroc=+0.64 ece=-0.50
MHIST         100%   C=0.01  auroc=+0.79 ece=-0.29   C=0.1   auroc=+1.00 ece=+0.29   C=10.0  auroc=+0.71 ece=+0.43   C=100.0 auroc=+0.71 ece=-0.64
LungHist700     1%   C=0.01  auroc=+1.00 ece=+0.79   C=0.1   auroc=+1.00 ece=+0.93   C=10.0  auroc=+1.00 ece=+0.71   C=100.0 auroc=+0.93 ece=+0.00
LungHist700    10%   C=0.01  auroc=+0.93 ece=-0.50   C=0.1   auroc=+0.93 ece=+0.71   C=10.0  auroc=+0.79 ece=-0.07   C=100.0 auroc=+0.64 ece=+0.21
LungHist700   100%   C=0.01  auroc=+0.43 ece=+0.14   C=0.1   auroc=+0.71 ece=+0.21   C=10.0  auroc=+0.93 ece=+0.07   C=100.0 auroc=+0.71 ece=-0.36
BRACS           1%   C=0.01  auroc=+0.93 ece=+0.50   C=0.1   a